![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)

# <h1><center>Tutorial  <a href="https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/python/tutorial.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook provides an introductory tutorial for [**MuJoCo** physics](https://github.com/google-deepmind/mujoco#readme), using the native Python bindings.

<!-- Copyright 2021 DeepMind Technologies Limited

     Licensed under the Apache License, Version 2.0 (the "License");
     you may not use this file except in compliance with the License.
     You may obtain a copy of the License at

         http://www.apache.org/licenses/LICENSE-2.0

     Unless required by applicable law or agreed to in writing, software
     distributed under the License is distributed on an "AS IS" BASIS,
     WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
     See the License for the specific language governing permissions and
     limitations under the License.
-->

# All imports

In [1]:
%load_ext autoreload
%autoreload 2

# Set up GPU rendering.
# import distutils.util
import os
import subprocess
# # if subprocess.run('nvidia-smi').returncode:
# #   raise RuntimeError(
# #       'Cannot communicate with GPU. '
# #       'Make sure you are using a GPU Colab runtime. '
# #       'Go to the Runtime menu and select Choose runtime type.')

# # Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# # This is usually installed as part of an Nvidia driver package, but the Colab
# # kernel doesn't install its driver via APT, and as a result the ICD is missing.
# # (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
# NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
# if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
#   with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
#     f.write("""{
#     "file_format_version" : "1.0.0",
#     "ICD" : {
#         "library_path" : "libEGL_nvidia.so.0"
#     }
# }
# """)

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

# Check if installation was succesful.
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
  import mujoco.viewer # Also have to import this to trigger the installation of the viewer.
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Other imports and helper functions
import time
import itertools
import numpy as np
# from scipy.spatial.transform import Rotation as Robj
from scipy.optimize import curve_fit, fsolve, least_squares
from scipy.signal import medfilt
from scipy.stats import linregress
from utils.helper_fns import *
from utils.render_opts import RendererViewerOpts
import utils.robot_controller as robot_controller
from load_obj_in_env import load_environment, load_photoshoot
# import utils.com_estimation as com
from utils.com_estimation import tau_model, tau_app_model, theta_from_tau, align_zeros

# Graphics and plotting.
import mediapy as media
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import contextlib

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)
# Set matplotlib font size
fonts = {'size' : 20}
plt.rc('font', **fonts)
%matplotlib inline
# %matplotlib notebook

Setting environment variable to use GPU rendering:
env: MUJOCO_GL=egl
Checking that the installation succeeded:
Installation successful.


## Full Toppling Simulation

In [5]:

# ======================== Toggle visualization here =========================
VIZ = True   # set to False to record video without showing the viewer
# ============================================================================

## Let's recall the model to reset the simulation
# 0: box_exp, 10: heart, 11: L_shape, 12: monitor, 13: soda, 14: flashlight
OBJECT = 14

model, data = load_environment(num="lekiwi", launch_viewer=False)
## ===================================================================

run_duration = 16.0  # seconds

## Additions for video recording
rv = RendererViewerOpts(model, data, vis=VIZ)

# ===========================================================================
with rv: # enters viewer if vis=True, sets viewer opts, and readies offscreen renderer for video capture
    while rv.viewer_is_running() and data.time < run_duration: #and not irb.stop:
        mujoco.mj_step(model, data)                 # Step the simulation
        rv.sync()                               # Update the interactive viewer (if present)
        rv.capture_frame_if_due(data)

# No need to close renderer since context manager handles it

print('\n\n======================================')
print(f'Simulation ended in t = {data.time:.2f} seconds.')
print('======================================')

# Play the movie
media.show_video(rv.frames, fps=rv.framerate)




Simulation ended in t = 16.00 seconds.


This browser does not support the video tag.
